# Effect A + B -> C

主线：同一 Qwen base 上分别训练 A/B 的 QLoRA-DPO adapter，做精确 LoRA delta 相加，再在 held-out C 上评估。先完整运行主线；Joint DPO 和 SFT 是可选消融。

In [ ]:
import subprocess
from google.colab import drive

subprocess.run(["nvidia-smi"], check=True)
drive.mount("/content/drive")

## 获取仓库并安装

如果 HTTPS clone 因私有仓库失败，请通过 Colab GitHub 授权打开仓库，或把 checkout 放到 Drive，然后修改 `REPO_DIR`。不要把 GitHub token 写进 notebook。

In [ ]:
import sys
from pathlib import Path

REPO_DIR = Path("/content/HumanStudy-Bench")
if not (REPO_DIR / ".git").exists():
    subprocess.run([
        "git", "clone", "--branch", "pipeline", "--single-branch",
        "https://github.com/HumanStudy-Hub/HumanStudy-Bench.git",
        str(REPO_DIR),
    ], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-r",
    str(REPO_DIR / "effect_algebra" / "requirements-colab.txt"),
], check=True)
print("repo:", REPO_DIR)

In [ ]:
MODEL = "Qwen/Qwen2.5-7B-Instruct"  # T4 16GB 可统一改成 Qwen/Qwen2.5-3B-Instruct
ROOT = Path("/content/drive/MyDrive/effect_algebra_ab_c")
DATA = ROOT / "data"
ADAPTERS = ROOT / "adapters"
RESULTS = ROOT / "results"
for path in (DATA, ADAPTERS, RESULTS):
    path.mkdir(parents=True, exist_ok=True)

def run_module(module, *args):
    command = [sys.executable, "-m", module, *[str(arg) for arg in args]]
    print("$", " ".join(command), flush=True)
    subprocess.run(command, cwd=REPO_DIR, check=True)

def evaluate_suite(label, adapter=None):
    args = [
        "--model-label", label,
        "--base-model", MODEL,
        "--dataset", f"A_test={DATA / 'eval' / 'A_test.jsonl'}",
        "--dataset", f"B_test={DATA / 'eval' / 'B_test.jsonl'}",
        "--dataset", f"B_control={DATA / 'eval' / 'B_no_feedback_control.jsonl'}",
        "--dataset", f"C_test={DATA / 'eval' / 'C_test.jsonl'}",
        "--output-dir", RESULTS / label,
    ]
    if adapter is not None:
        args.extend(["--adapter", adapter])
    run_module("effect_algebra.evaluate_suite", *args)

## 生成数据并 fail-closed 验证

期望：`errors=0`、`C_used_for_training=false`。

In [ ]:
run_module(
    "effect_algebra.build_datasets",
    "--repo-root", REPO_DIR,
    "--output-dir", DATA,
)
run_module("effect_algebra.validate_datasets", "--data-dir", DATA)

## Base pre-test

先保存未训练 base 的 A/B/C 行为，后面所有提升都相对这个基线解释。

In [ ]:
evaluate_suite("base")

## 分别训练 A 和 B

两个命令在独立子进程中运行，前一个结束后 GPU 权重会释放。B 的完整 30 轮历史若超过 token limit，训练会直接失败而不是裁剪。

In [ ]:
run_module(
    "effect_algebra.train_dpo",
    "--train-file", DATA / "dpo" / "A_train.jsonl",
    "--eval-file", DATA / "dpo" / "A_dev.jsonl",
    "--output-dir", ADAPTERS / "A_dpo",
    "--run-name", "effect-A-dpo",
    "--base-model", MODEL,
)
run_module(
    "effect_algebra.train_dpo",
    "--train-file", DATA / "dpo" / "B_train.jsonl",
    "--eval-file", DATA / "dpo" / "B_dev.jsonl",
    "--output-dir", ADAPTERS / "B_dpo",
    "--run-name", "effect-B-dpo",
    "--base-model", MODEL,
)

## 精确合并 Delta_A + Delta_B

In [ ]:
run_module(
    "effect_algebra.merge_adapters",
    "--adapter-a", ADAPTERS / "A_dpo",
    "--adapter-b", ADAPTERS / "B_dpo",
    "--output-dir", ADAPTERS / "A_plus_B_cat",
    "--base-model", MODEL,
    "--weight-a", "1.0",
    "--weight-b", "1.0",
    "--combination-type", "cat",
)

## 统一评估 A-only、B-only、A+B

每个 adapter 只加载一次模型，连续评估 A/B/B-control/C。

In [ ]:
evaluate_suite("A_only", ADAPTERS / "A_dpo")
evaluate_suite("B_only", ADAPTERS / "B_dpo")
evaluate_suite("A_plus_B", ADAPTERS / "A_plus_B_cat")

## 可选：Joint A+B DPO baseline

只有 A-only 和 B-only 各自在自己的 test 上成功后才运行。

In [ ]:
RUN_JOINT = False
if RUN_JOINT:
    run_module(
        "effect_algebra.train_dpo",
        "--train-file", DATA / "dpo" / "AB_train.jsonl",
        "--eval-file", DATA / "dpo" / "AB_dev.jsonl",
        "--output-dir", ADAPTERS / "AB_joint_dpo",
        "--run-name", "effect-AB-joint-dpo",
        "--base-model", MODEL,
    )
    evaluate_suite("AB_joint", ADAPTERS / "AB_joint_dpo")

## 可选：LoRA-SFT baseline

这不是另一种 LoRA；它仅把 objective 从 DPO 换成 chosen-answer SFT。

In [ ]:
RUN_SFT_BASELINE = False
if RUN_SFT_BASELINE:
    for effect in ("A", "B"):
        run_module(
            "effect_algebra.train_sft",
            "--train-file", DATA / "dpo" / f"{effect}_train.jsonl",
            "--eval-file", DATA / "dpo" / f"{effect}_dev.jsonl",
            "--output-dir", ADAPTERS / f"{effect}_sft",
            "--run-name", f"effect-{effect}-sft",
            "--base-model", MODEL,
        )
        evaluate_suite(f"{effect}_sft", ADAPTERS / f"{effect}_sft")

## 汇总结果

CSV 和 Markdown 会保存在 Drive。C 同时查看 normative accuracy、human probability MAE 和 authority alignment，不能压缩成一个总分。

In [ ]:
result_files = sorted(RESULTS.glob("*/*.json"))
result_files = [path for path in result_files if path.name != "suite_manifest.json"]
run_module(
    "effect_algebra.compare_results",
    *result_files,
    "--csv", ROOT / "comparison.csv",
    "--markdown", ROOT / "comparison.md",
)
print("Artifacts:", ROOT)